[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [PyMongo and Beanie, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pymongo-and-beanie-deep-dive.html)

# Saving Changes


## What you will be able to do

Change a document through the model four ways, and say which of them writes the whole thing and
which writes only what changed. Turn on state management, ask a document what has changed since it
was read, and save only that. Turn on revisions, and have a write refused when somebody else got
there first. And recognize the quiet one, which is `save` on a document you read a moment ago
putting back every field as it was then, including the ones somebody else has since corrected.


## The idea

### The problem

`save` is the obvious method and it is a whole-document write. It sends every field, so it undoes
every change made between your read and your write, and reports success.

That is exactly the `replace_one` trap of **Update Operators**, arriving through an API that looks
like it is doing something smaller.

### What the four ways are

`save` writes the whole document. `save_changes` writes only the fields that changed, and needs
state management turned on to know which those are. `set` and `inc` are operations you send without
having read anything, which is the safest and least convenient.

### Why state management is off by default

Because remembering the document as it was read costs memory on every document, and most programs
do not need it. It is one line in `Settings`, and `save_changes` raises without it rather than
guessing.

### Where this shows up

Any handler that reads an object, changes a field and saves it, which is most of them. The failure
is invisible until two people use the system at once, and then it is a bug report about a setting
that keeps resetting itself.

### What this notebook covers

`save`, `save_changes`, `set`, `inc` and `update`. `use_state_management` and `get_changes`.
`use_revision`, and the conflict it turns into an error. Then the two exceptions, the silent
overwrite, and the linked document that was never saved.

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import asyncio

from beanie import Document, init_beanie
from pymongo import AsyncMongoClient


class Item(Document):
    sku: str
    name: str
    stock: int

    class Settings:
        name = "saving"
        use_state_management = True


async def main():
    client = AsyncMongoClient("mongodb://127.0.0.1:27017/shop")
    await init_beanie(database=client.get_default_database(), document_models=[Item])

    await Item.delete_all()
    await Item(sku="A-1", name="a thing", stock=1).insert()

    mine = await Item.find_one(Item.sku == "A-1")          # I read it
    await Item.find_one(Item.sku == "A-1").set({Item.name: "renamed by someone else"})

    mine.stock = 99
    await mine.save()                                      # and write the whole thing back

    now = await Item.find_one(Item.sku == "A-1")
    print("stock:", now.stock, "| name:", now.name)
    print("the other edit is gone, and nothing reported a conflict")
    await client.close()


asyncio.run(main())
```

```
stock: 99 | name: a thing
the other edit is gone, and nothing reported a conflict
```

One field was changed in Python and every field was written to MongoDB. The rename that happened in
between is gone, the write reported success, and nothing anywhere in that program could have told
you.


## Setup

Ten imports, MongoDB, and the boot cell.

- `beanie` with `Document` and `init_beanie`, and the module itself for its exception classes
- `pymongo` provides `AsyncMongoClient`, and the boot cell uses its synchronous client
- `subprocess`, `os`, `sys`, `time`, `random`, `version` and `PackageNotFoundError` run the boot cell

There are no helpers in this notebook: every section defines the model it needs, because the whole
subject is what the model's `Settings` turn on.


In [1]:
import os
import random
import subprocess
import sys
import time
from importlib.metadata import PackageNotFoundError, version

try:
    if version("pymongo") != "4.18.1" or version("beanie") != "2.2.0":
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "pymongo==4.18.1", "beanie==2.2.0"], check=True)

import beanie
import pymongo
from beanie import Document, init_beanie
from pymongo import AsyncMongoClient

DBPATH = "/content/mongo" if os.path.isdir("/content") else "/tmp/guide_mongo/rs"
LOGPATH = f"{DBPATH}.log"
URI = "mongodb://127.0.0.1:27017/shop"                              # no credential, anywhere
PUBLISHED = ["jammy", "noble"]                                      # codenames MongoDB builds for


def shell(command):
    """Run a shell command and hand back what it printed, without letting it stop the notebook."""
    done = subprocess.run(command, shell=True, capture_output=True, text=True)
    return done.returncode, (done.stdout + done.stderr).strip()


def answering(timeout=2000):
    """Whether a mongod is there, asked directly rather than through topology discovery."""
    try:
        with pymongo.MongoClient("mongodb://127.0.0.1:27017/?directConnection=true",
                                 serverSelectionTimeoutMS=timeout) as client:
            client.admin.command("ping")
            return True
    except pymongo.errors.PyMongoError:
        return False


def install_server():
    """Add MongoDB's own apt repository and install the server package. Linux only."""
    if shell("which mongod")[0] == 0:
        return "already installed"

    codename = shell("lsb_release -cs")[1]
    if codename not in PUBLISHED:                                   # an unpublished one breaks apt
        print(f"  Ubuntu '{codename}' has no MongoDB repository; using '{PUBLISHED[-1]}' instead")
        codename = PUBLISHED[-1]

    if not shell("grep -o avx /proc/cpuinfo | head -1")[1]:
        raise RuntimeError("This CPU has no AVX. Every MongoDB build since 5.0 needs it, so "
                           "neither the apt package nor the tarball will start here.")

    sudo = "" if os.geteuid() == 0 else "sudo "
    shell(f"curl -fsSL https://www.mongodb.org/static/pgp/server-8.0.asc "
          f"| {sudo}gpg --dearmor -o /usr/share/keyrings/mongodb-8.0.gpg")
    shell(f'echo "deb [signed-by=/usr/share/keyrings/mongodb-8.0.gpg] '
          f'https://repo.mongodb.org/apt/ubuntu {codename}/mongodb-org/8.0 multiverse" '
          f'| {sudo}tee /etc/apt/sources.list.d/mongodb-8.0.list')
    shell(f"{sudo}apt-get -qq update "                              # this one list file only
          f"-o Dir::Etc::sourcelist=sources.list.d/mongodb-8.0.list "
          f"-o Dir::Etc::sourceparts=-")
    code, out = shell(f"{sudo}apt-get -qq -y install mongodb-org-server")
    if shell("which mongod")[0] != 0:
        raise RuntimeError(f"mongodb-org-server did not install. apt said: {out[-400:]}")
    return f"installed from the {codename} repository"


def start_server(wait=30):
    """Start mongod with a replica set name, idempotently. Returns what it had to do."""
    if answering():
        return "already running"
    if sys.platform != "linux":
        raise RuntimeError("No mongod is answering on 127.0.0.1:27017. Start your own server "
                           "with --replSet rs0 and run this again: this cell only installs one "
                           "on Linux, which is what Colab runs.")

    print(" ", install_server())
    os.makedirs(DBPATH, exist_ok=True)
    code, out = shell(f"mongod --dbpath {DBPATH} --replSet rs0 --bind_ip 127.0.0.1 "
                      f"--fork --logpath {LOGPATH}")
    if code != 0:                                                   # --fork hides the reason
        print("  mongod did not start. The last lines of its log:")
        print("   ", shell(f"tail -20 {LOGPATH}")[1].replace("\n", "\n    "))
        raise RuntimeError("mongod exited. The log above says why.")

    for attempt in range(1, wait + 1):
        if answering():
            return "installed and started"
        print(f"  waiting for mongod ({attempt})")
        time.sleep(1)
    raise RuntimeError(f"mongod did not answer within {wait} seconds.")

def initiate(wait=30):
    """Make the single node a replica set, which is what transactions and migrations need."""
    with pymongo.MongoClient("mongodb://127.0.0.1:27017/?directConnection=true",
                             serverSelectionTimeoutMS=2000) as boot:
        try:                                                        # an explicit host, not getHostName()
            boot.admin.command("replSetInitiate",
                               {"_id": "rs0", "members": [{"_id": 0, "host": "127.0.0.1:27017"}]})
        except pymongo.errors.OperationFailure as error:
            if error.code != 23:                                    # 23 is AlreadyInitialized
                raise

        for attempt in range(1, wait + 1):
            hello = boot.admin.command("hello")
            if hello.get("isWritablePrimary"):
                return f"replica set {hello['setName']}, primary"
            time.sleep(1)
    raise RuntimeError(f"No primary after {wait} seconds. The last hello was: {hello}")

SIZE = 500                                                          # Indexes and the catalog raise this
KINDS = ["laptop", "monitor", "keyboard", "mouse", "cable"]
MAKERS = ["Aster", "Belden", "Corvid", "Dalgo"]


def seed(size=None, force=False):
    """Fill shop.products and shop.reviews, once, from a fixed seed so every run agrees."""
    size = SIZE if size is None else size
    client = pymongo.MongoClient(URI, tz_aware=True)
    shop = client.get_default_database()

    if not force and shop.products.estimated_document_count() == size:
        client.close()
        return size

    shop.products.drop()
    shop.reviews.drop()
    random.seed(0)                                                  # the whole reason runs agree

    products, reviews = [], []
    for number in range(size):
        kind = KINDS[number % len(KINDS)]
        product = {
            "_id": number,
            "sku": f"{kind[:3].upper()}-{number:06d}",
            "name": f"{MAKERS[number % len(MAKERS)]} {kind} {number}",
            "maker": MAKERS[number % len(MAKERS)],
            "kind": kind,
            "price": round(random.uniform(5, 2000), 2),
            "stock": random.randint(0, 400),
            "tags": sorted(random.sample(["sale", "new", "refurbished", "bulk", "clearance"], 2)),
            "size": {"w": random.randint(5, 60), "h": random.randint(2, 40)},
        }
        products.append(product)
        for _ in range(random.randint(0, 3)):
            reviews.append({"product_id": number, "stars": random.randint(1, 5),
                            "body": f"A review of {product['name']}"})

    for start in range(0, len(products), 5000):                     # batches, not one huge insert
        shop.products.insert_many(products[start:start + 5000])
    for start in range(0, len(reviews), 5000):
        shop.reviews.insert_many(reviews[start:start + 5000])

    client.close()
    return size


def report():
    """One line naming what this notebook is running against."""
    with pymongo.MongoClient(URI, tz_aware=True) as client:
        build = client.admin.command("buildInfo")["version"].split(".")[0]
        shop = client.get_default_database()
        return (f"MongoDB {build} | pymongo {version('pymongo')} | beanie {version('beanie')} "
                f"| products: {shop.products.count_documents({})}")


print("server: ", start_server())
print("replica:", initiate())
print("seeded: ", seed(), "products")
print(report())


server:  already running
replica: replica set rs0, primary
seeded:  500 products
MongoDB 8 | pymongo 4.18.1 | beanie 2.2.0 | products: 500


## Worked examples

### Four ways to change a document


In [2]:
class Item(Document):
    sku: str
    name: str
    stock: int

    class Settings:
        name = "saving"
        use_state_management = True                                 # needed for save_changes


client = AsyncMongoClient(URI)
await init_beanie(database=client.get_default_database(), document_models=[Item])

await Item.delete_all()
await Item(sku="A-1", name="a thing", stock=10).insert()
print("starting from:", (await Item.find_one(Item.sku == "A-1")).model_dump(exclude={"id"}))


starting from: {'sku': 'A-1', 'name': 'a thing', 'stock': 10}


`save`, with the whole document:


In [3]:
item = await Item.find_one(Item.sku == "A-1")
item.stock = 11
await item.save()
print("after save():       ", (await Item.find_one(Item.sku == "A-1")).stock)


after save():        11


`save_changes`, with only what changed:


In [4]:
item = await Item.find_one(Item.sku == "A-1")
item.stock = 12
print("what it will write: ", item.get_changes())
await item.save_changes()
print("after save_changes():", (await Item.find_one(Item.sku == "A-1")).stock)


what it will write:  {'stock': 12}
after save_changes(): 12


`get_changes()` is the whole reason state management exists: the document remembers what it looked
like when it was read, so it can send a `$set` of the difference rather than the lot.

And `set` and `inc`, which change the document without reading it at all:


In [5]:
await Item.find_one(Item.sku == "A-1").set({Item.name: "renamed"})
await Item.find_one(Item.sku == "A-1").inc({Item.stock: 5})

now = await Item.find_one(Item.sku == "A-1")
print("after set and inc:  ", now.stock, "|", now.name)


after set and inc:   17 | renamed


These are the safe ones. `inc` in particular is the `$inc` of **Update Operators**: the arithmetic
happens on the server, so two processes incrementing at once both land.

### What each one actually sends

The difference is not stylistic:


In [6]:
item = await Item.find_one(Item.sku == "A-1")
item.stock = 20

print("save() would send:       every field:", sorted(item.model_dump(exclude={"id"})))
print("save_changes() would send:", item.get_changes())
print("set() sends exactly what you pass it, and needs no read at all")


save() would send:       every field: ['name', 'sku', 'stock']
save_changes() would send: {'stock': 20}
set() sends exactly what you pass it, and needs no read at all


`save` sends `sku`, `name` and `stock`. `save_changes` sends `{"stock": 20}`. On this document the
difference is nothing; on a document with a large embedded array it is the difference between
writing a few bytes and rewriting megabytes.

### The overwrite

Two readers, one of whom writes last:


In [7]:
await Item.find_one(Item.sku == "A-1").set({Item.name: "the correct name", Item.stock: 10})

mine = await Item.find_one(Item.sku == "A-1")                       # I read it
await Item.find_one(Item.sku == "A-1").set({Item.name: "corrected by somebody else"})

mine.stock = 42
await mine.save()

after = await Item.find_one(Item.sku == "A-1")
print("with save():        ", after.stock, "|", after.name)


with save():         42 | the correct name


The correction is gone. Now the same sequence with `save_changes`:


In [8]:
await Item.find_one(Item.sku == "A-1").set({Item.name: "the correct name", Item.stock: 10})

mine = await Item.find_one(Item.sku == "A-1")
await Item.find_one(Item.sku == "A-1").set({Item.name: "corrected by somebody else"})

mine.stock = 42
await mine.save_changes()

after = await Item.find_one(Item.sku == "A-1")
print("with save_changes():", after.stock, "|", after.name, "<- both survived")


with save_changes(): 42 | corrected by somebody else <- both survived


Both changes are there, because `save_changes` wrote a `$set` naming only `stock` and left `name`
alone. That is the argument for turning state management on: it is one line and it turns a
whole-document write into a targeted one.

### Revisions, for when losing is not acceptable

`save_changes` avoids clobbering a field nobody touched. It does not help when two people change
**the same** field. For that there is `use_revision`:


In [9]:
class Counted(Document):
    sku: str
    stock: int

    class Settings:
        name = "saving_revised"
        use_state_management = True
        use_revision = True                                         # every write checks it


await init_beanie(database=client.get_default_database(), document_models=[Item, Counted])
await Counted.delete_all()
await Counted(sku="R-1", stock=1).insert()

one = await Counted.find_one(Counted.sku == "R-1")
two = await Counted.find_one(Counted.sku == "R-1")                  # the same document, twice

one.stock = 2
await one.save_changes()
print("the first write went through:", (await Counted.find_one(Counted.sku == "R-1")).stock)

two.stock = 3
try:
    await two.save_changes()
except beanie.exceptions.RevisionIdWasChanged as error:
    print("the second was refused:", type(error).__name__, f"with message {str(error)!r}")


the first write went through: 2
the second was refused: RevisionIdWasChanged with message ''


Beanie keeps a `revision_id` on the document, writes a new one each time, and makes every write
conditional on the one it read. The second writer's condition no longer matches, so nothing is
written and you are told.

The cost is that you now have to decide what to do about it, which is the point: a conflict is a
decision, not an error to log.

### When to reach for which

| What you want | How to write it |
|---|---|
| to change a field you just read | `document.field = x` then `await document.save_changes()` |
| to change a field without reading | `await Model.find_one(...).set({Model.field: x})` |
| to add to a number | `await Model.find_one(...).inc({Model.field: 1})` |
| to write the document entire | `await document.save()`, deliberately |
| to know what changed | `document.get_changes()` |
| to be told about a conflict | `use_revision = True` in `Settings` |
| to change many at once | `await Model.find(...).set({...})` |

The default is `set` and `inc` when you know what you want to change, and `save_changes` when you
have a document in hand. `save` is for when you genuinely mean "this document is now exactly this",
which is rarer than the method's name suggests.

### A stock adjustment that cannot lose an edit, finished


In [10]:
async def adjust(sku, by, note=None):
    """Change the count on the server, and the note only if one was given."""
    query = Counted.find_one(Counted.sku == sku)
    if await query.count() == 0:
        return None
    await Counted.find_one(Counted.sku == sku).inc({Counted.stock: by})
    return (await Counted.find_one(Counted.sku == sku)).stock


async def rename_carefully(sku, name):
    """Read, change one field, and write only that field back."""
    item = await Item.find_one(Item.sku == sku)
    if item is None:
        return None
    item.name = name
    await item.save_changes()
    return item.get_changes()


await Counted.delete_all()
await Counted(sku="R-2", stock=10).insert()

print("after +5: ", await adjust("R-2", 5))
print("after -3: ", await adjust("R-2", -3))
print("missing:  ", await adjust("NOPE", 1))

print("rename:   ", await rename_carefully("A-1", "a careful rename"))
print("stored:   ", (await Item.find_one(Item.sku == "A-1")).name)


after +5:  15
after -3:  12
missing:   None
rename:    {}
stored:    a careful rename


`adjust` never reads the count into Python, so two of them running at once both land and neither can
overwrite the other. `rename_carefully` does read the document, because it needs to know it exists,
and then writes back only the field it touched.

`get_changes()` is empty after the save, which is how the document says it has nothing outstanding.

### Where each part came from

| In the adjustment | What it relies on | The section that showed it |
|---|---|---|
| `inc({Counted.stock: by})` | arithmetic on the server | Four ways to change a document |
| never reading the count | two writers both landing | The overwrite |
| `save_changes()` | state management knowing the difference | Four ways to change a document |
| `get_changes()` | the document remembering how it was read | What each one actually sends |
| `use_state_management` | `save_changes` raising without it | Common errors |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pymongo-and-beanie-deep-dive/13-saving-changes-solutions.ipynb).

**1.** Change a field and write it back with `save_changes`.


In [11]:
# your code here


**2.** Print `get_changes()` before and after saving.


In [12]:
# your code here


**3.** Change a field with `set`, without reading the document.


In [13]:
# your code here


**4.** Add to a number with `inc`.


In [14]:
# your code here


**5.** Show `save` losing somebody else's edit and `save_changes` keeping it.


In [15]:
# your code here


**6.** Make two writers conflict on a model with `use_revision`.


In [16]:
# your code here


## Common errors

### beanie.exceptions.StateManagementIsTurnedOff


In [17]:
class Plain(Document):
    sku: str
    stock: int

    class Settings:
        name = "saving_plain"                                       # no use_state_management


await init_beanie(database=client.get_default_database(),
                  document_models=[Item, Counted, Plain])
await Plain.delete_all()
await Plain(sku="P-1", stock=1).insert()

plain = await Plain.find_one(Plain.sku == "P-1")
plain.stock = 2
try:
    await plain.save_changes()
except beanie.exceptions.StateManagementIsTurnedOff as error:
    print(f"{type(error).__module__}.{type(error).__name__}: {error}")


beanie.exceptions.StateManagementIsTurnedOff: State management is turned off for this document


Without state management the document does not remember what it looked like when it was read, so
there is no difference to compute and Beanie says so rather than falling back to `save`, which
would have been a much worse answer.

One line in `Settings` fixes it, and there is very little reason not to have it on:


In [18]:
print("Item has it:  ", Item.get_settings().use_state_management)
print("Plain does not:", Plain.get_settings().use_state_management)
print()
print("with it on, save_changes writes:", (await Item.find_one(Item.sku == "A-1")).get_changes())


Item has it:   True
Plain does not: False

with it on, save_changes writes: {}


### beanie.exceptions.RevisionIdWasChanged


In [19]:
await Counted.delete_all()
await Counted(sku="R-3", stock=1).insert()

first = await Counted.find_one(Counted.sku == "R-3")
second = await Counted.find_one(Counted.sku == "R-3")

first.stock = 10
await first.save_changes()

second.stock = 20
try:
    await second.save_changes()
except beanie.exceptions.RevisionIdWasChanged as error:
    print(f"{type(error).__module__}.{type(error).__name__}, message {str(error)!r}")
    print("stored:", (await Counted.find_one(Counted.sku == "R-3")).stock)


beanie.exceptions.RevisionIdWasChanged, message ''
stored: 10


The message is empty, which makes the class name the whole of the information, and the important
part is what did **not** happen: the second write was not applied.

What to do about it is yours to choose. Re-read and retry is the usual answer, and it is only
correct when the change can be recomputed from the new state:


In [20]:
async def retry_once(sku, amount):
    """Re-read and apply the change again, which is safe because it is relative."""
    for attempt in (1, 2):
        document = await Counted.find_one(Counted.sku == sku)
        document.stock += amount
        try:
            await document.save_changes()
            return attempt, document.stock
        except beanie.exceptions.RevisionIdWasChanged:
            continue
    raise RuntimeError("gave up")


print("retried:", await retry_once("R-3", 5))


retried: (1, 15)


### No error: save putting back what it read


In [21]:
await Item.find_one(Item.sku == "A-1").set({Item.name: "before", Item.stock: 1})

stale = await Item.find_one(Item.sku == "A-1")                      # read now
await Item.find_one(Item.sku == "A-1").set({Item.name: "changed meanwhile"})

stale.stock = 2
await stale.save()                                                  # written later

result = await Item.find_one(Item.sku == "A-1")
print("name:", result.name, "| stock:", result.stock)
print("the write succeeded and the rename was reverted")


name: before | stock: 2
the write succeeded and the rename was reverted


This is the first look again, and it is here because it is the one failure in this notebook that
never raises. A long-lived object, an edit form, a background job holding a document while it works:
anything that reads early and saves late will do this.

`save_changes` is the fix in almost every case, and `use_revision` is the fix when even that is not
enough:


In [22]:
await Item.find_one(Item.sku == "A-1").set({Item.name: "before", Item.stock: 1})

stale = await Item.find_one(Item.sku == "A-1")
await Item.find_one(Item.sku == "A-1").set({Item.name: "changed meanwhile"})

stale.stock = 2
await stale.save_changes()

result = await Item.find_one(Item.sku == "A-1")
print("name:", result.name, "| stock:", result.stock, "<- both edits kept")


name: changed meanwhile | stock: 2 <- both edits kept


### No error: the field that was never marked as changed


In [23]:
item = await Item.find_one(Item.sku == "A-1")
print("changes before touching anything:", item.get_changes())

item.name = item.name                                               # assigned the same value
print("after assigning the same value: ", item.get_changes())

item.name = "actually different"
print("after a real change:            ", item.get_changes())
await item.save_changes()
print("after saving:                   ", item.get_changes())


changes before touching anything: {}
after assigning the same value:  {}
after a real change:             {'name': 'actually different'}
after saving:                    {}


State management compares values rather than watching assignments, so writing the same value back
is correctly not a change. That is usually what you want and it is worth knowing, because it means
`save_changes` on a document you have not modified sends nothing at all rather than touching the
document.

`get_changes()` is empty again after the save, which is how the object records that it is in step
with the database.


In [24]:
for model in (Item, Counted, Plain):
    await model.delete_all()
await client.close()
print("tidied up and closed")


tidied up and closed


## Recap

- `save` writes the whole document and silently undoes anything changed between your read and your
  write. It is `replace_one` wearing a friendlier name.
- `save_changes` writes only the fields that differ from what was read, and needs
  `use_state_management = True` in `Settings` or it raises `StateManagementIsTurnedOff`.
- `get_changes()` is what it will send. It is empty after a save, and empty when a value was
  assigned back to itself.
- `set` and `inc` send an operation without reading anything, which is the safest option and the
  one to reach for when you already know what you want to change.
- `use_revision = True` makes every write conditional on the revision that was read, so a second
  writer gets `RevisionIdWasChanged` instead of winning. The message is empty; the class name is
  the information.
- A conflict is a decision. Re-read and retry works when the change is relative, and needs thought
  when it is not.


## What is next

**Link and BackLink** is how Beanie refers to another document: the `Link` that is not the document
until you fetch it, `fetch_links` replacing a round trip per document with one `$lookup`, and the
nesting depth past which links quietly come back unresolved.


---

&#8592; **Previous:** [Async Queries](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pymongo-and-beanie-deep-dive/12-async-queries.ipynb)  &nbsp;·&nbsp;  [PyMongo and Beanie, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pymongo-and-beanie-deep-dive.html)
